# Building Image Generation Applications (OpenAI)

There's more to LLMs than text generation. You can also generate images from text descriptions. Images as a modality are useful across MedTech, architecture, tourism, game development, marketing, and more. In this lesson we work with today's **GPT Image** models on the OpenAI platform.

## Learning goals

By the end of this lesson you'll be able to:

- Explain what image generation is and where it's useful.
- Understand the `gpt-image` model family and how it differs from the legacy DALL·E models.
- Build an image generation application and edit images.

## What is image generation?

Image generation models create images from a text prompt. Modern models such as `gpt-image` learn the relationship between text and images during training, then iteratively turn random noise into an image that matches your prompt.

Two well-known families of image models are:

- **`gpt-image` (OpenAI)** — the current generation used in this lesson. It supports text-to-image generation and image editing (inpainting with a mask).
- **Midjourney** — a popular third-party model with its own service and Discord-based workflow.

> The older OpenAI image models — **DALL·E 2** and **DALL·E 3** — are legacy. DALL·E 3 is no longer available for new deployments, and features like `create_variation` only existed in DALL·E 2. Use the `gpt-image` models for new applications.

> **Important:** `gpt-image` models return the generated image as **base64** (`b64_json`), not as a URL. Your code decodes the base64 string to bytes and saves it — there's no image URL to download.


## Izdelava vaše prve aplikacije za generiranje slik

Kaj torej potrebujete za izdelavo aplikacije za generiranje slik? Potrebujete naslednje knjižnice:

- **python-dotenv**, močno priporočamo uporabo te knjižnice za shranjevanje vaših skrivnosti v datoteko *.env* ločeno od kode.
- **openai**, ta knjižnica se uporablja za interakcijo z OpenAI API-jem.
- **pillow**, za delo s slikami v Pythonu.


1. Ustvarite datoteko *.env* z naslednjo vsebino:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Zberite zgornje knjižnice v datoteko z imenom *requirements.txt* na naslednji način:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Nato ustvarite virtualno okolje in namestite knjižnice:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Za Windows uporabite naslednje ukaze za ustvarjanje in aktiviranje vašega virtualnega okolja:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Dodajte naslednjo kodo v datoteko z imenom *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Create OpenAI object (reads OPENAI_API_KEY from your .env)
    client = openai.OpenAI()


    try:
        # Create an image by using the image generation API
        generation_response = client.images.generate(
            model="gpt-image-2",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
            size='1024x1024',
            n=1
        )
        # Set the directory for the stored image
        image_dir = os.path.join(os.curdir, 'images')

        # If the directory doesn't exist, create it
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Initialize the image path (note the filetype should be png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image models return the image as base64 (b64_json), not a URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Display the image in the default image viewer
        image = Image.open(image_path)
        image.show()

    # catch exceptions
    except openai.BadRequestError as err:
        print(err)

    ```

Razložimo to kodo:

- Najprej uvozimo potrebne knjižnice, vključno z OpenAI knjižnico, dotenv knjižnico, modul base64 in Pillow knjižnico.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- Nato ustvarimo klienta, ki prebere API ključ iz vaše ``.env``.

    ```python
    # Create OpenAI object
    client = openai.OpenAI()
    ```

- Naslednji korak je generiranje slike:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-2",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image` modeli vrnejo sliko kot **base64** niz v `data[0].b64_json`. Dekodiramo jo v bajte in zapišemo v datoteko — ne obstaja URL za prenos.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Na koncu odprete sliko in uporabite privzet pregledovalnik slik za prikaz:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Več podrobnosti o generiranju slike

Poglejmo parametre `images.generate`:

- **model**, je slikovni model, ki ga uporabljamo (na primer `gpt-image-2`).
- **prompt**, je besedilni poziv za generiranje slike. Tukaj je "Zajček na konju, ki drži liziko, na megleni travnati jasi, kjer rastejo trobentice".
- **size**, je velikost generirane slike (na primer `1024x1024`, `1536x1024`, `1024x1536` ali `"auto"`).
- **n**, je število generiranih slik. Tukaj generiramo eno.

> Slikovni modeli ne sprejmejo parametra `temperature` — to je kontrola za generiranje besedila. Za večjo raznolikost pokličite API znova; za manjšo raznolikost pa naredite svoj poziv bolj specifičen.

## Dodatne zmožnosti generiranja slik

Videli ste, kako generirati sliko z nekaj vrsticami kode v Pythonu. `gpt-image` modeli lahko tudi **urejajo** obstoječo sliko. Z zagotovitvijo slike, neobvezne **maske** (ki označi območje za spremembo) in poziva lahko spremenite del slike — na primer dodate klobuk našemu zajčku.

```python
response = client.images.edit(
  model="gpt-image-2",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# edits are also returned as base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Osnovna slika vsebuje le zajca; končna slika ima klobuk na zajcu.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
